In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jammjace/train-csv/train.csv


# imports

In [2]:
import os
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    classification_report
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)


In [3]:
SEED = 42

DISTILBERT_NAME = "distilbert-base-uncased"
DEBERTA_NAME = "microsoft/deberta-v3-small"

PILOT_TRAIN_SIZE = 6000
PILOT_VAL_SIZE = 1500

# Original fixed truncation baseline
PROMPT_MAX = 32
RESPONSE_MAX = 108

# Selected truncation configuration
MAX_LENGTH = 256
PROMPT_CAP = 64
MIN_RESPONSE_TOKENS = 64

LEARNING_RATE = 1e-5
BATCH_SIZE = 16
EPOCHS = 2

set_seed(SEED)


# load dataset

In [4]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "train.csv":
            print(os.path.join(root, file))

/kaggle/input/datasets/jammjace/train-csv/train.csv


In [5]:
train = pd.read_csv(
    "/kaggle/input/datasets/jammjace/train-csv/train.csv"
)

train.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


# EDA

In [6]:
# print("Shape:", train.shape)

# print("\nColumns:")
# print(train.columns.tolist())

# train.info()

In [7]:
# train.isnull().sum().sort_values(ascending=False)

In [8]:
# print(
#     "Duplicate rows:",
#     train.duplicated().sum()
# )

In [9]:
# target_cols = [
#     "winner_model_a",
#     "winner_model_b",
#     "winner_tie"
# ]

# train[target_cols].sum()

In [10]:
# train[target_cols].mean()

In [11]:
# train[target_cols].sum().plot(
#     kind="bar",
#     figsize=(7, 4)
# )

# plt.ylabel("Number of samples")
# plt.title("Target Distribution")
# plt.xticks(rotation=0)

# plt.show()

In [12]:
# winner_count = train[target_cols].sum(axis=1)

# winner_count.value_counts()

In [13]:
train["label"] = np.select(
    [
        train["winner_model_a"] == 1,
        train["winner_model_b"] == 1,
        train["winner_tie"] == 1
    ],
    [
        0,
        1,
        2
    ],
    default=-1
)

In [14]:
label_map = {
    0: "Model A",
    1: "Model B",
    2: "Tie"
}

train["label_name"] = (
    train["label"]
    .map(label_map)
)

In [15]:
train["label_name"].value_counts()

label_name
Model A    20064
Model B    19652
Tie        17761
Name: count, dtype: int64

In [16]:
train["label_name"].value_counts(
    normalize=True
)

label_name
Model A    0.349079
Model B    0.341911
Tie        0.309011
Name: proportion, dtype: float64

# Text EDA

In [17]:
text_cols = [
    "prompt",
    "response_a",
    "response_b"
]

for col in text_cols:
    train[col] = (
        train[col]
        .fillna("")
        .astype(str)
    )

In [18]:
sample = train.iloc[0]

print("PROMPT:")
print(sample["prompt"])

print("\nRESPONSE A:")
print(sample["response_a"])

print("\nRESPONSE B:")
print(sample["response_b"])

print("\nWINNER:")
print(sample["label_name"])

PROMPT:
["Is it morally right to try to have a certain percentage of females on managerial positions?","OK, does pineapple belong on a pizza? Relax and give me fun answer."]

RESPONSE A:
["The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.\n\nHere are some arguments in favor of and against such policies:\n\n**Arguments in favor:**\n\n1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.\n\n2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and society as a whole.\n\n3. *

In [19]:
for col in text_cols:
    train[f"{col}_char_len"] = (
        train[col]
        .str.len()
    )

In [20]:
train[
    [
        "prompt_char_len",
        "response_a_char_len",
        "response_b_char_len"
    ]
].describe()

,prompt_char_len,response_a_char_len,response_b_char_len
count,57477.000000,57477.000000,57477.000000
mean,369.198584,1377.818919,1386.162169
std,1073.341472,1513.932333,1537.800488
min,7.000000,4.000000,4.000000
25%,52.000000,408.000000,413.000000
50%,96.000000,1076.000000,1086.000000
75%,243.000000,1862.000000,1873.000000
max,33056.000000,54058.000000,53830.000000


In [21]:
# plt.figure(figsize=(8, 5))

# plt.hist(
#     train["response_a_char_len"],
#     bins=50,
#     alpha=0.5,
#     label="Response A"
# )

# plt.hist(
#     train["response_b_char_len"],
#     bins=50,
#     alpha=0.5,
#     label="Response B"
# )

# plt.xlabel("Character length")
# plt.ylabel("Number of responses")
# plt.title("Response Length Distribution")
# plt.legend()

# plt.show()

In [22]:
# plt.figure(figsize=(8, 5))

# plt.hist(
#     np.log1p(
#         train["response_a_char_len"]
#     ),
#     bins=50,
#     alpha=0.5,
#     label="Response A"
# )

# plt.hist(
#     np.log1p(
#         train["response_b_char_len"]
#     ),
#     bins=50,
#     alpha=0.5,
#     label="Response B"
# )

# plt.xlabel("log(1 + character length)")
# plt.ylabel("Number of responses")
# plt.title("Log Response Length Distribution")
# plt.legend()

# plt.show()

In [23]:
# train["response_length_diff"] = (
#     train["response_a_char_len"]
#     - train["response_b_char_len"]
# )

In [24]:
# train.groupby("label_name")[
#     "response_length_diff"
# ].agg(
#     ["mean", "median", "std"]
# )

# combined text for TF-IDF baseline

In [25]:
train["combined_text"] = (
    "Prompt:\n"
    + train["prompt"]
    + "\n\nResponse A:\n"
    + train["response_a"]
    + "\n\nResponse B:\n"
    + train["response_b"]
)

# train val test split

In [26]:
train_val_df, test_df = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["label"]
)

In [27]:
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.25,
    random_state=42,
    stratify=train_val_df["label"]
)

In [28]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 34485
Validation: 11496
Test: 11496


# TF-IDF + LR baseline

In [29]:
# tfidf = TfidfVectorizer(
#     max_features=50000,
#     ngram_range=(1, 2),
#     min_df=2,
#     max_df=0.95,
#     sublinear_tf=True
# )

using combined text only for baseline. for distilbert, use three fields separately.

In [30]:
# X_train_tfidf = tfidf.fit_transform(
#     train_df["combined_text"]
# )

# X_val_tfidf = tfidf.transform(
#     val_df["combined_text"]
# )

# X_test_tfidf = tfidf.transform(
#     test_df["combined_text"]
# )

In [31]:
# lr = LogisticRegression(
#     max_iter=1000
# )

# lr.fit(
#     X_train_tfidf,
#     train_df["label"]
# )

In [32]:
# val_pred = lr.predict(
#     X_val_tfidf
# )

# val_prob = lr.predict_proba(
#     X_val_tfidf
# )

In [33]:
# print(
#     "Validation Accuracy:",
#     accuracy_score(
#         val_df["label"],
#         val_pred
#     )
# )

In [34]:
# print(
#     "Validation Log Loss:",
#     log_loss(
#         val_df["label"],
#         val_prob,
#         labels=[0, 1, 2]
#     )
# )

In [35]:
# print(
#     classification_report(
#         val_df["label"],
#         val_pred,
#         target_names=[
#             "Model A",
#             "Model B",
#             "Tie"
#         ]
#     )
# )

# create small transformer pilot dataset

In [36]:
# train_small, _ = train_test_split(
#     train_df,
#     train_size=PILOT_TRAIN_SIZE,
#     random_state=SEED,
#     stratify=train_df["label"]
# )

# val_small, _ = train_test_split(
#     val_df,
#     train_size=PILOT_VAL_SIZE,
#     random_state=SEED,
#     stratify=val_df["label"]
# )


# load distilbert

In [37]:
# Model names are defined once in the configuration cell above.


In [38]:
distilbert_tokenizer = AutoTokenizer.from_pretrained(
    DISTILBERT_NAME
)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [39]:
set_seed(SEED)

# distilbert_model = AutoModelForSequenceClassification.from_pretrained(
#     DISTILBERT_NAME,
#     num_labels=3
# )


# check GPU

In [40]:
print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

CUDA available: True
GPU: Tesla T4


# balanced truncation

In [41]:
# PROMPT_MAX and RESPONSE_MAX are defined in the configuration cell.


In [42]:
# def tokenize_balanced(example, tokenizer):
#     prompt_ids = tokenizer(
#         example["prompt"],
#         add_special_tokens=False,
#         truncation=True,
#         max_length=PROMPT_MAX
#     )["input_ids"]

#     response_a_ids = tokenizer(
#         example["response_a"],
#         add_special_tokens=False,
#         truncation=True,
#         max_length=RESPONSE_MAX
#     )["input_ids"]

#     response_b_ids = tokenizer(
#         example["response_b"],
#         add_special_tokens=False,
#         truncation=True,
#         max_length=RESPONSE_MAX
#     )["input_ids"]

#     input_ids = (
#         [tokenizer.cls_token_id]
#         + prompt_ids
#         + [tokenizer.sep_token_id]
#         + response_a_ids
#         + [tokenizer.sep_token_id]
#         + response_b_ids
#         + [tokenizer.sep_token_id]
#     )

#     return {
#         "input_ids": input_ids,
#         "attention_mask": [1] * len(input_ids)
#     }


# convert pilot data to hugging face dataset

In [43]:
# hf_train_small = Dataset.from_pandas(
#     train_small[
#         [
#             "prompt",
#             "response_a",
#             "response_b",
#             "label"
#         ]
#     ],
#     preserve_index=False
# )

In [44]:
# hf_val_small = Dataset.from_pandas(
#     val_small[
#         [
#             "prompt",
#             "response_a",
#             "response_b",
#             "label"
#         ]
#     ],
#     preserve_index=False
# )

# tokenize pilot dataset

In [45]:
# tokenized_train_distilbert_baseline = hf_train_small.map(
#     lambda x: tokenize_balanced(x, distilbert_tokenizer)
# )


In [46]:
# tokenized_val_distilbert_baseline = hf_val_small.map(
#     lambda x: tokenize_balanced(x, distilbert_tokenizer)
# )


In [47]:
# lengths = [
#     len(x)
#     for x in tokenized_train_distilbert_baseline["input_ids"]
# ]

# print("Maximum:", max(lengths))
# print("Median:", np.median(lengths))
# print("Mean:", np.mean(lengths))


# dynamic padding

In [48]:
# distilbert_collator = DataCollatorWithPadding(
#     tokenizer=distilbert_tokenizer
# )


# eval function

In [49]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    exp_logits = np.exp(
        logits
        - np.max(
            logits,
            axis=1,
            keepdims=True
        )
    )

    probabilities = (
        exp_logits
        / exp_logits.sum(
            axis=1,
            keepdims=True
        )
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    loss = log_loss(
        labels,
        probabilities,
        labels=[0, 1, 2]
    )

    return {
        "accuracy": accuracy,
        "log_loss": loss
    }

# training config

In [50]:
# distilbert_baseline_args = TrainingArguments(
#     output_dir="/kaggle/working/distilbert_pilot",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=1,
#     weight_decay=0.01,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="log_loss",
#     greater_is_better=False,
#     logging_steps=50,
#     report_to="none",
#     fp16=torch.cuda.is_available(),
#     seed=SEED,
#     data_seed=SEED
# )


# trainer

In [51]:
# distilbert_trainer = Trainer(
#     model=distilbert_model,
#     args=distilbert_baseline_args,
#     train_dataset=tokenized_train_distilbert_baseline,
#     eval_dataset=tokenized_val_distilbert_baseline,
#     data_collator=distilbert_collator,
#     compute_metrics=compute_metrics
# )


# fine-tune distilbert

In [52]:
# distilbert_trainer.train()


- TF-IDF + Logistic Regression
  - Accuracy: 0.3713
  - Log loss: 1.1143
- DistilBERT pilot
  - Accuracy: 0.4040
  - Log loss: 1.0850

transformer improved by about:
- +3.3 percentage points accuracy
- 0.029 lower log loss

# use DeBERTa-v3-small 

In [53]:
# deberta_tokenizer = AutoTokenizer.from_pretrained(
#     DEBERTA_NAME,
#     use_fast=False
# )


The same `tokenize_balanced(example, tokenizer)` function is used for both DistilBERT and DeBERTa so each model always uses its own tokenizer.


In [54]:
# tokenized_train_deberta = hf_train_small.map(
#     lambda x: tokenize_balanced(x, deberta_tokenizer)
# )

# tokenized_val_deberta = hf_val_small.map(
#     lambda x: tokenize_balanced(x, deberta_tokenizer)
# )


In [55]:
# deberta_collator = DataCollatorWithPadding(
#     tokenizer=deberta_tokenizer
# )


In [56]:
# set_seed(SEED)

# deberta_model = AutoModelForSequenceClassification.from_pretrained(
#     DEBERTA_NAME,
#     num_labels=3,
#     torch_dtype=torch.float32
# )

# deberta_model = deberta_model.float()


In [57]:
# deberta_args = TrainingArguments(
#     output_dir="/kaggle/working/deberta_pilot",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=1,
#     weight_decay=0.01,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="log_loss",
#     greater_is_better=False,
#     logging_steps=50,
#     report_to="none",
#     fp16=False,
#     bf16=False,
#     seed=SEED,
#     data_seed=SEED
# )


In [58]:
# deberta_trainer = Trainer(
#     model=deberta_model,
#     args=deberta_args,
#     train_dataset=tokenized_train_deberta,
#     eval_dataset=tokenized_val_deberta,
#     data_collator=deberta_collator,
#     compute_metrics=compute_metrics
# )


**Implementation note:** FP16 training produced an AMP gradient-unscaling error for DeBERTa in this Kaggle environment. The pilot therefore uses FP32 for DeBERTa.


In [59]:
# Consolidated into the DeBERTa setup above.


In [60]:
# Consolidated into the DeBERTa setup above.


turning off fp16 due to mixed-precision training

In [61]:
# Consolidated into the DeBERTa setup above.


In [62]:
# Consolidated into the DeBERTa setup above.


In [63]:
# deberta_trainer.train()


- DistilBERT currently has the best accuracy.
- DistilBERT and DeBERTa have almost identical log loss:
    - DistilBERT: 1.084950
    - DeBERTa: 1.085024

- That difference in log loss is negligible.
- DeBERTa is slower:
    - validation runtime ~11.3 s
    - DistilBERT ~6.7 s

# inspect both models 
- model archi
- inputs
- learning during fine-tuning

In [64]:
# print(distilbert_model)


In [65]:
# DistilBERT variables are named explicitly from the start.


In [66]:
# print(deberta_model.config)

DeBERTa uses relative position attention, disentangled attention and separate handling of content and positive info. 

inspect actual tokenized inputs for both models

In [67]:
# %whos

In [68]:
# tokenized_train_distilbert = tokenized_train_distilbert_baseline
# tokenized_val_distilbert = tokenized_val_distilbert_baseline


In [69]:
# DeBERTa variables are already named explicitly.


In [70]:
# # Pick one example
# example = hf_val_small[0]

# print("ORIGINAL INPUT")
# print("=" * 80)

# print("\nPROMPT:")
# print(example["prompt"])

# print("\nRESPONSE A:")
# print(example["response_a"])

# print("\nRESPONSE B:")
# print(example["response_b"])

# print("\nTRUE LABEL:")
# print(example["label"])

In [71]:
# tokenize_balanced(example, tokenizer) was defined once above.


In [72]:
# # for distilbert
# distil_encoded = tokenize_balanced(
#     example,
#     distilbert_tokenizer
# )

In [73]:
# # for deberta
# deberta_encoded = tokenize_balanced(
#     example,
#     deberta_tokenizer
# )

In [74]:
# tokenized_train_distilbert = tokenized_train_distilbert_baseline
# tokenized_val_distilbert = tokenized_val_distilbert_baseline


In [75]:
# DeBERTa tokenized datasets were already created above.


In [76]:
# print(
#     distilbert_tokenizer.decode(
#         tokenized_val_distilbert[0]["input_ids"]
#     )
# )

# print("\n" + "=" * 80 + "\n")

# print(
#     deberta_tokenizer.decode(
#         tokenized_val_deberta[0]["input_ids"]
#     )
# )

In [77]:
# # DistilBERT
# distil_ids = tokenized_val_distilbert[0]["input_ids"]
# distil_tokens = distilbert_tokenizer.convert_ids_to_tokens(distil_ids)

# print("DistilBERT token count:", len(distil_tokens))
# print(distil_tokens)

In [78]:
# # DeBERTa
# deberta_ids = tokenized_val_deberta[0]["input_ids"]
# deberta_tokens = deberta_tokenizer.convert_ids_to_tokens(deberta_ids)

# print("DeBERTa token count:", len(deberta_tokens))
# print(deberta_tokens)

In [79]:
# print("DISTILBERT")
# print(distilbert_tokenizer.decode(distil_ids))

# print("\n" + "=" * 80 + "\n")

# print("DEBERTA")
# print(deberta_tokenizer.decode(deberta_ids))

- both hit 252 token budget
- preserve the full prompt
- both truncate response A and B

key difference: deberta tokenizes the example more efficiently

With the same 108 token response budget, DeBERTa preserves slightly more actual text.

tokenization strategy: 
- distilbert uses wordpiece
    - subwords are marked with ##
- deberta uses sentencepiece
    - _ roughly means new word starts here
 
deberta has a more efficient tokenizer for this text, but that did not translate into better pilot accuracy

# quantify truncation problem across validation set

check whether sentence-aware truncation would matter broadly.
- Compare for DistilBERT:
  - percentage of prompts truncated
  - percentage of Response A truncated
  - percentage of Response B truncated
  - average tokens lost
- Do the same for DeBERTa.
  
Then run a controlled experiment:
- same 6000 / 1500 pilot split
- same model
- same epochs
- same learning rate
- only change truncation strategy

In [80]:
# import pandas as pd
# import numpy as np

# def analyze_truncation(dataset, tokenizer):

#     results = []

#     for example in dataset:

#         # Full token counts before truncation
#         prompt_full = len(
#             tokenizer(
#                 example["prompt"],
#                 add_special_tokens=False
#             )["input_ids"]
#         )

#         a_full = len(
#             tokenizer(
#                 example["response_a"],
#                 add_special_tokens=False
#             )["input_ids"]
#         )

#         b_full = len(
#             tokenizer(
#                 example["response_b"],
#                 add_special_tokens=False
#             )["input_ids"]
#         )

#         # Number of tokens retained by current strategy
#         prompt_kept = min(prompt_full, PROMPT_MAX)
#         a_kept = min(a_full, RESPONSE_MAX)
#         b_kept = min(b_full, RESPONSE_MAX)

#         results.append({
#             "prompt_full": prompt_full,
#             "a_full": a_full,
#             "b_full": b_full,

#             "prompt_kept": prompt_kept,
#             "a_kept": a_kept,
#             "b_kept": b_kept,

#             "prompt_truncated": prompt_full > PROMPT_MAX,
#             "a_truncated": a_full > RESPONSE_MAX,
#             "b_truncated": b_full > RESPONSE_MAX,

#             "prompt_tokens_lost": max(0, prompt_full - PROMPT_MAX),
#             "a_tokens_lost": max(0, a_full - RESPONSE_MAX),
#             "b_tokens_lost": max(0, b_full - RESPONSE_MAX)
#         })

#     return pd.DataFrame(results)

In [81]:
# %whos

In [82]:
# distilbert_tokenizer = AutoTokenizer.from_pretrained(
#     DISTILBERT_NAME
# )

# deberta_tokenizer = AutoTokenizer.from_pretrained(
#     DEBERTA_NAME,
#     use_fast=False
# )


In [83]:
# %whos

# truncation analysis

In [84]:
# def analyze_truncation(dataset, tokenizer):

#     results = []

#     for example in dataset:

#         prompt_full = len(
#             tokenizer(
#                 example["prompt"],
#                 add_special_tokens=False
#             )["input_ids"]
#         )

#         a_full = len(
#             tokenizer(
#                 example["response_a"],
#                 add_special_tokens=False
#             )["input_ids"]
#         )

#         b_full = len(
#             tokenizer(
#                 example["response_b"],
#                 add_special_tokens=False
#             )["input_ids"]
#         )

#         results.append({
#             "prompt_full": prompt_full,
#             "a_full": a_full,
#             "b_full": b_full,

#             "prompt_truncated": prompt_full > PROMPT_MAX,
#             "a_truncated": a_full > RESPONSE_MAX,
#             "b_truncated": b_full > RESPONSE_MAX,

#             "prompt_tokens_lost": max(0, prompt_full - PROMPT_MAX),
#             "a_tokens_lost": max(0, a_full - RESPONSE_MAX),
#             "b_tokens_lost": max(0, b_full - RESPONSE_MAX)
#         })

#     return pd.DataFrame(results)

In [85]:
# distil_truncation = analyze_truncation(
#     hf_val_small,
#     distilbert_tokenizer
# )

# deberta_truncation = analyze_truncation(
#     hf_val_small,
#     deberta_tokenizer
# )

In [86]:
# def truncation_summary(df, model_name):

#     print(model_name)
#     print("=" * 60)

#     print(f"Prompt truncated: {df['prompt_truncated'].mean():.1%}")
#     print(f"Response A truncated: {df['a_truncated'].mean():.1%}")
#     print(f"Response B truncated: {df['b_truncated'].mean():.1%}")

#     print("\nAverage full token counts:")
#     print("Prompt:", df["prompt_full"].mean())
#     print("Response A:", df["a_full"].mean())
#     print("Response B:", df["b_full"].mean())

#     print("\nAverage tokens lost:")
#     print("Prompt:", df["prompt_tokens_lost"].mean())
#     print("Response A:", df["a_tokens_lost"].mean())
#     print("Response B:", df["b_tokens_lost"].mean())

In [87]:
# truncation_summary(distil_truncation, "DistilBERT")
# print()
# truncation_summary(deberta_truncation, "DeBERTa")

# experiment 1 
sentence-aware truncatoin, dynamic allocation between prompt/A/B, larger overall sequence budget, fairness between A and B

- use one total 384 token budget
- preserve complete sentences where possible
- dynamically share space across prompt, A, and B
- alternate between A and B so one response cannot dominate
- fall back to token truncation if a single sentence itself is too long

In [88]:
# DYNAMIC_MAX_LENGTH = 384

# def split_sentences(text):
#     text = str(text).strip()

#     sentences = re.split(
#         r'(?<=[.!?])\s+|\\n+|\n+',
#         text
#     )

#     return [
#         sentence.strip()
#         for sentence in sentences
#         if sentence.strip()
#     ]


# def get_ids(text, tokenizer):
#     return tokenizer(
#         text,
#         add_special_tokens=False
#     )["input_ids"]


In [89]:
# get_ids() is defined in the helper cell above.


In [90]:
# def tokenize_dynamic_sentence_aware(
#     example,
#     tokenizer,
#     max_length=DYNAMIC_MAX_LENGTH
# ):

#     prompt_sentences = split_sentences(example["prompt"])
#     a_sentences = split_sentences(example["response_a"])
#     b_sentences = split_sentences(example["response_b"])

#     prompt_kept = []
#     a_kept = []
#     b_kept = []

#     # 4 special tokens:
#     # [CLS] prompt [SEP] A [SEP] B [SEP]
#     available_tokens = max_length - 4

#     # --------------------------------------------------
#     # 1. Add prompt
#     # --------------------------------------------------

#     for sentence in prompt_sentences:

#         sentence_ids = get_ids(sentence, tokenizer)

#         if len(sentence_ids) <= available_tokens:
#             prompt_kept.append(sentence)
#             available_tokens -= len(sentence_ids)

#         else:
#             # If nothing from prompt fits as a complete sentence,
#             # preserve as much of this sentence as possible.
#             if len(prompt_kept) == 0 and available_tokens > 0:
#                 partial_ids = sentence_ids[:available_tokens]

#                 prompt_kept.append(
#                     tokenizer.decode(
#                         partial_ids,
#                         skip_special_tokens=True
#                     )
#                 )

#                 available_tokens = 0

#             break

#     # --------------------------------------------------
#     # 2. Alternate between A and B
#     # --------------------------------------------------

#     a_index = 0
#     b_index = 0

#     while available_tokens > 0:

#         added = False

#         # Try next A sentence
#         if a_index < len(a_sentences):

#             sentence = a_sentences[a_index]
#             sentence_ids = get_ids(sentence, tokenizer)

#             if len(sentence_ids) <= available_tokens:
#                 a_kept.append(sentence)
#                 available_tokens -= len(sentence_ids)
#                 a_index += 1
#                 added = True

#         # Try next B sentence
#         if b_index < len(b_sentences):

#             sentence = b_sentences[b_index]
#             sentence_ids = get_ids(sentence, tokenizer)

#             if len(sentence_ids) <= available_tokens:
#                 b_kept.append(sentence)
#                 available_tokens -= len(sentence_ids)
#                 b_index += 1
#                 added = True

#         if not added:
#             break

#     # --------------------------------------------------
#     # 3. Convert final text into token IDs
#     # --------------------------------------------------

#     prompt_ids = get_ids(
#         " ".join(prompt_kept),
#         tokenizer
#     )

#     a_ids = get_ids(
#         " ".join(a_kept),
#         tokenizer
#     )

#     b_ids = get_ids(
#         " ".join(b_kept),
#         tokenizer
#     )

#     input_ids = (
#         [tokenizer.cls_token_id]
#         + prompt_ids
#         + [tokenizer.sep_token_id]
#         + a_ids
#         + [tokenizer.sep_token_id]
#         + b_ids
#         + [tokenizer.sep_token_id]
#     )

#     attention_mask = [1] * len(input_ids)

#     return {
#         "input_ids": input_ids,
#         "attention_mask": attention_mask
#     }

In [91]:
# example = hf_val_small[0]

# dynamic_example = tokenize_dynamic_sentence_aware(
#     example,
#     distilbert_tokenizer
# )

# print("Token count:", len(dynamic_example["input_ids"]))

# print(
#     distilbert_tokenizer.decode(
#         dynamic_example["input_ids"]
#     )
# )

regenerate pilot dataset

In [92]:
# tokenized_train_distilbert_dynamic = hf_train_small.map(
#     lambda x: tokenize_dynamic_sentence_aware(
#         x,
#         distilbert_tokenizer
#     )
# )

# tokenized_val_distilbert_dynamic = hf_val_small.map(
#     lambda x: tokenize_dynamic_sentence_aware(
#         x,
#         distilbert_tokenizer
#     )
# )

check sequence-length distribution

In [93]:
# dynamic_lengths = [
#     len(x["input_ids"])
#     for x in tokenized_train_distilbert_dynamic
# ]

# print("Maximum:", max(dynamic_lengths))
# print("Mean:", np.mean(dynamic_lengths))
# print("Median:", np.median(dynamic_lengths))
# print("90th percentile:", np.percentile(dynamic_lengths, 90))

load fresh distilbert 

In [94]:
# distilbert_dynamic_model = (
#     AutoModelForSequenceClassification.from_pretrained(
#         DISTILBERT_NAME,
#         num_labels=3
#     )
# )


In [95]:
# distilbert_dynamic_collator = DataCollatorWithPadding(
#     tokenizer=distilbert_tokenizer
# )

In [96]:
# distilbert_dynamic_args = TrainingArguments(
#     output_dir="/kaggle/working/distilbert_dynamic_384",

#     learning_rate=2e-5,

#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,

#     num_train_epochs=1,

#     weight_decay=0.01,

#     eval_strategy="epoch",
#     save_strategy="epoch",

#     load_best_model_at_end=True,
#     metric_for_best_model="log_loss",
#     greater_is_better=False,

#     logging_steps=50,
#     report_to="none",

#     fp16=torch.cuda.is_available(),
#     seed=SEED,
#     data_seed=SEED
# )


In [97]:
# compute_metrics() was defined once in the evaluation section above.


In [98]:
# distilbert_dynamic_trainer = Trainer(
#     model=distilbert_dynamic_model,
#     args=distilbert_dynamic_args,

#     train_dataset=tokenized_train_distilbert_dynamic,
#     eval_dataset=tokenized_val_distilbert_dynamic,

#     data_collator=distilbert_dynamic_collator,

#     compute_metrics=compute_metrics
# )

In [99]:
# distilbert_dynamic_trainer.train()

In [100]:
# dynamic_results = distilbert_dynamic_trainer.evaluate(
#     tokenized_val_distilbert_dynamic
# )

# dynamic_results

experiment did not improve DistilBERT

check token allocation

In [101]:
# def analyze_dynamic_allocation(dataset, tokenizer):

#     rows = []

#     for example in dataset:

#         encoded = tokenize_dynamic_sentence_aware(
#             example,
#             tokenizer
#         )

#         # Re-run the same internal logic to recover section lengths
#         prompt_sentences = split_sentences(example["prompt"])
#         a_sentences = split_sentences(example["response_a"])
#         b_sentences = split_sentences(example["response_b"])

#         prompt_kept = []
#         a_kept = []
#         b_kept = []

#         available_tokens = DYNAMIC_MAX_LENGTH - 4

#         # prompt
#         for sentence in prompt_sentences:
#             sentence_ids = get_ids(sentence, tokenizer)

#             if len(sentence_ids) <= available_tokens:
#                 prompt_kept.append(sentence)
#                 available_tokens -= len(sentence_ids)
#             else:
#                 if len(prompt_kept) == 0 and available_tokens > 0:
#                     partial_ids = sentence_ids[:available_tokens]
#                     prompt_kept.append(
#                         tokenizer.decode(
#                             partial_ids,
#                             skip_special_tokens=True
#                         )
#                     )
#                     available_tokens = 0
#                 break

#         # A/B alternating
#         a_index = 0
#         b_index = 0

#         while available_tokens > 0:
#             added = False

#             if a_index < len(a_sentences):
#                 ids = get_ids(a_sentences[a_index], tokenizer)

#                 if len(ids) <= available_tokens:
#                     a_kept.append(a_sentences[a_index])
#                     available_tokens -= len(ids)
#                     a_index += 1
#                     added = True

#             if b_index < len(b_sentences):
#                 ids = get_ids(b_sentences[b_index], tokenizer)

#                 if len(ids) <= available_tokens:
#                     b_kept.append(b_sentences[b_index])
#                     available_tokens -= len(ids)
#                     b_index += 1
#                     added = True

#             if not added:
#                 break

#         prompt_tokens = len(
#             get_ids(" ".join(prompt_kept), tokenizer)
#         )

#         a_tokens = len(
#             get_ids(" ".join(a_kept), tokenizer)
#         )

#         b_tokens = len(
#             get_ids(" ".join(b_kept), tokenizer)
#         )

#         rows.append({
#             "prompt_tokens": prompt_tokens,
#             "a_tokens": a_tokens,
#             "b_tokens": b_tokens,
#             "total_tokens": len(encoded["input_ids"]),
#             "unused_tokens": DYNAMIC_MAX_LENGTH - len(encoded["input_ids"])
#         })

#     return pd.DataFrame(rows)

In [102]:
# dynamic_allocation = analyze_dynamic_allocation(
#     hf_val_small,
#     distilbert_tokenizer
# )

In [103]:
# print(dynamic_allocation.describe())

In [104]:
# print("Average allocation")
# print("------------------")
# print("Prompt:", dynamic_allocation["prompt_tokens"].mean())
# print("Response A:", dynamic_allocation["a_tokens"].mean())
# print("Response B:", dynamic_allocation["b_tokens"].mean())
# print("Total:", dynamic_allocation["total_tokens"].mean())
# print("Unused:", dynamic_allocation["unused_tokens"].mean())

# print("\nMedian allocation")
# print("------------------")
# print("Prompt:", dynamic_allocation["prompt_tokens"].median())
# print("Response A:", dynamic_allocation["a_tokens"].median())
# print("Response B:", dynamic_allocation["b_tokens"].median())

In [105]:
# dynamic_allocation["a_b_difference"] = (
#     dynamic_allocation["a_tokens"]
#     - dynamic_allocation["b_tokens"]
# )

# print(
#     dynamic_allocation["a_b_difference"].describe()
# )

dynamic strategy problems:
- Too many unused tokens
  - Average unused: 70.35
  - Median total length: 370, so many examples use the budget well.
  - But the mean drops to 313.65 because some examples waste huge amounts.
  - Maximum unused: 362 tokens.
- A/B allocation is unstable
  - Mean difference is only +4.6, which looks balanced overall.
  - But SD is 81 tokens.
  - Extreme cases range from:
    - A getting 337 fewer tokens than B
    - A getting 336 more tokens than B

# experiment 2
change the truncation strategy:
- cap prompt length
- allocate response budget proportionally to original A/B token lengths
- enforce a minimum for both A and B
- preserve full sentences where possible
- use remaining space instead of leaving large unused gaps

In [106]:
# Reuse split_sentences() and get_ids() defined above.
# Current selected defaults are defined in the global configuration cell.


In [107]:
# def truncate_sentence_aware(text, tokenizer, token_budget):

#     if token_budget <= 0:
#         return []

#     sentences = split_sentences(text)

#     kept_ids = []

#     for sentence in sentences:

#         sentence_ids = get_ids(sentence, tokenizer)

#         # keep whole sentence if it fits
#         if len(kept_ids) + len(sentence_ids) <= token_budget:
#             kept_ids.extend(sentence_ids)

#         else:
#             # fill remaining space with part of next sentence
#             remaining = token_budget - len(kept_ids)

#             if remaining > 0:
#                 kept_ids.extend(sentence_ids[:remaining])

#             break

#     return kept_ids

In [108]:
# def tokenize_proportional_sentence_aware(
#     example,
#     tokenizer,
#     max_length=MAX_LENGTH
# ):

#     # Full untruncated token lengths
#     prompt_full_ids = get_ids(
#         example["prompt"],
#         tokenizer
#     )

#     a_full_ids = get_ids(
#         example["response_a"],
#         tokenizer
#     )

#     b_full_ids = get_ids(
#         example["response_b"],
#         tokenizer
#     )

#     # --------------------------------------------------
#     # 1. Prompt allocation
#     # --------------------------------------------------

#     prompt_budget = min(
#         len(prompt_full_ids),
#         PROMPT_CAP
#     )

#     prompt_ids = truncate_sentence_aware(
#         example["prompt"],
#         tokenizer,
#         prompt_budget
#     )

#     # 4 special tokens:
#     # [CLS] prompt [SEP] A [SEP] B [SEP]
#     response_budget = (
#         max_length
#         - 4
#         - len(prompt_ids)
#     )

#     # --------------------------------------------------
#     # 2. Minimum allocation to both responses
#     # --------------------------------------------------

#     a_min = min(
#         MIN_RESPONSE_TOKENS,
#         len(a_full_ids)
#     )

#     b_min = min(
#         MIN_RESPONSE_TOKENS,
#         len(b_full_ids)
#     )

#     base_required = a_min + b_min

#     # Safety case
#     if base_required > response_budget:

#         scale = response_budget / base_required

#         a_budget = int(a_min * scale)
#         b_budget = response_budget - a_budget

#     else:

#         remaining = response_budget - base_required

#         # --------------------------------------------------
#         # 3. Allocate remaining budget proportionally
#         # --------------------------------------------------

#         total_length = len(a_full_ids) + len(b_full_ids)

#         if total_length == 0:
#             a_share = 0.5

#         else:
#             a_share = (
#                 len(a_full_ids)
#                 / total_length
#             )

#         a_extra = int(
#             remaining * a_share
#         )

#         b_extra = remaining - a_extra

#         a_budget = a_min + a_extra
#         b_budget = b_min + b_extra

#     # Never allocate more than response actually contains
#     a_budget = min(
#         a_budget,
#         len(a_full_ids)
#     )

#     b_budget = min(
#         b_budget,
#         len(b_full_ids)
#     )

#     # --------------------------------------------------
#     # 4. Redistribute unused budget
#     # --------------------------------------------------

#     used_budget = a_budget + b_budget

#     leftover = response_budget - used_budget

#     if leftover > 0:

#         a_remaining = (
#             len(a_full_ids) - a_budget
#         )

#         b_remaining = (
#             len(b_full_ids) - b_budget
#         )

#         total_remaining = (
#             a_remaining + b_remaining
#         )

#         if total_remaining > 0:

#             add_a = min(
#                 a_remaining,
#                 int(
#                     leftover
#                     * a_remaining
#                     / total_remaining
#                 )
#             )

#             a_budget += add_a
#             leftover -= add_a

#             add_b = min(
#                 b_remaining,
#                 leftover
#             )

#             b_budget += add_b
#             leftover -= add_b

#             # final leftover back to A if possible
#             if leftover > 0:
#                 add_a = min(
#                     len(a_full_ids) - a_budget,
#                     leftover
#                 )

#                 a_budget += add_a

#     # --------------------------------------------------
#     # 5. Sentence-aware truncation
#     # --------------------------------------------------

#     a_ids = truncate_sentence_aware(
#         example["response_a"],
#         tokenizer,
#         a_budget
#     )

#     b_ids = truncate_sentence_aware(
#         example["response_b"],
#         tokenizer,
#         b_budget
#     )

#     # --------------------------------------------------
#     # 6. Final sequence
#     # --------------------------------------------------

#     input_ids = (
#         [tokenizer.cls_token_id]
#         + prompt_ids
#         + [tokenizer.sep_token_id]
#         + a_ids
#         + [tokenizer.sep_token_id]
#         + b_ids
#         + [tokenizer.sep_token_id]
#     )

#     attention_mask = [1] * len(input_ids)

#     return {
#         "input_ids": input_ids,
#         "attention_mask": attention_mask
#     }


In [109]:
# example = hf_val_small[0]

# exp2_example = tokenize_proportional_sentence_aware(
#     example,
#     distilbert_tokenizer
# )

# print(
#     "Token count:",
#     len(exp2_example["input_ids"])
# )

# print(
#     distilbert_tokenizer.decode(
#         exp2_example["input_ids"]
#     )
# )

In [110]:
# tokenized_train_distilbert_exp2 = hf_train_small.map(
#     lambda x: tokenize_proportional_sentence_aware(
#         x,
#         distilbert_tokenizer,
#         max_length=384
#     )
# )

# tokenized_val_distilbert_exp2 = hf_val_small.map(
#     lambda x: tokenize_proportional_sentence_aware(
#         x,
#         distilbert_tokenizer,
#         max_length=384
#     )
# )


In [111]:
# exp2_lengths = [
#     len(x["input_ids"])
#     for x in tokenized_train_distilbert_exp2
# ]

# print("Maximum:", max(exp2_lengths))
# print("Mean:", np.mean(exp2_lengths))
# print("Median:", np.median(exp2_lengths))
# print(
#     "90th percentile:",
#     np.percentile(exp2_lengths, 90)
# )

In [112]:
# distilbert_exp2_model = (
#     AutoModelForSequenceClassification.from_pretrained(
#         DISTILBERT_NAME,
#         num_labels=3
#     )
# )


In [113]:
# distilbert_exp2_collator = DataCollatorWithPadding(
#     tokenizer=distilbert_tokenizer
# )

In [114]:
# distilbert_exp2_args = TrainingArguments(
#     output_dir="/kaggle/working/distilbert_exp2",

#     learning_rate=2e-5,

#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,

#     num_train_epochs=1,

#     weight_decay=0.01,

#     eval_strategy="epoch",
#     save_strategy="epoch",

#     load_best_model_at_end=True,
#     metric_for_best_model="log_loss",
#     greater_is_better=False,

#     logging_steps=50,
#     report_to="none",

#     fp16=torch.cuda.is_available(),
#     seed=SEED,
#     data_seed=SEED
# )


In [115]:
# distilbert_exp2_trainer = Trainer(
#     model=distilbert_exp2_model,
#     args=distilbert_exp2_args,

#     train_dataset=tokenized_train_distilbert_exp2,
#     eval_dataset=tokenized_val_distilbert_exp2,

#     data_collator=distilbert_exp2_collator,

#     compute_metrics=compute_metrics
# )

In [116]:
# distilbert_exp2_trainer.train()

In [117]:
# exp2_results = distilbert_exp2_trainer.evaluate(
#     tokenized_val_distilbert_exp2
# )

# exp2_results

# experiment 3 
use experiment 2 tokenizer setup, change number of epochs

In [118]:
# distilbert_underfit_model = (
#     AutoModelForSequenceClassification.from_pretrained(
#         DISTILBERT_NAME,
#         num_labels=3
#     )
# )


In [119]:
# distilbert_underfit_args = TrainingArguments(
#     output_dir="/kaggle/working/distilbert_underfit_test",

#     learning_rate=2e-5,

#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,

#     num_train_epochs=3,

#     weight_decay=0.01,

#     eval_strategy="epoch",
#     save_strategy="epoch",

#     load_best_model_at_end=True,
#     metric_for_best_model="log_loss",
#     greater_is_better=False,

#     logging_steps=50,
#     report_to="none",

#     fp16=torch.cuda.is_available(),
#     seed=SEED,
#     data_seed=SEED
# )


In [120]:
# distilbert_underfit_trainer = Trainer(
#     model=distilbert_underfit_model,
#     args=distilbert_underfit_args,

#     train_dataset=tokenized_train_distilbert_exp2,
#     eval_dataset=tokenized_val_distilbert_exp2,

#     data_collator=distilbert_exp2_collator,

#     compute_metrics=compute_metrics
# )

In [121]:
# distilbert_underfit_trainer.train()

# experiment 4
extend to 5 epochs, use evaluation every epoch and let trainer keep the best checkpoint

In [122]:
# distilbert_5epoch_model = (
#     AutoModelForSequenceClassification.from_pretrained(
#         DISTILBERT_NAME,
#         num_labels=3
#     )
# )


In [123]:
# distilbert_5epoch_args = TrainingArguments(
#     output_dir="/kaggle/working/distilbert_5epoch",

#     learning_rate=2e-5,

#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,

#     num_train_epochs=5,

#     weight_decay=0.01,

#     eval_strategy="epoch",
#     save_strategy="epoch",

#     load_best_model_at_end=True,
#     metric_for_best_model="log_loss",
#     greater_is_better=False,

#     logging_steps=50,
#     report_to="none",

#     fp16=torch.cuda.is_available(),
#     seed=SEED,
#     data_seed=SEED
# )


In [124]:
# distilbert_5epoch_trainer = Trainer(
#     model=distilbert_5epoch_model,
#     args=distilbert_5epoch_args,

#     train_dataset=tokenized_train_distilbert_exp2,
#     eval_dataset=tokenized_val_distilbert_exp2,

#     data_collator=distilbert_exp2_collator,

#     compute_metrics=compute_metrics
# )

In [125]:
# distilbert_5epoch_trainer.train()

In [126]:
# distilbert_5epoch_results = distilbert_5epoch_trainer.evaluate(
#     tokenized_val_distilbert_exp2
# )

# distilbert_5epoch_results

overfitting starts after about epoch 2.

# experiment 5 
tune learning rate at 2 epochs

In [127]:
# def run_lr_experiment(lr_value):
#     set_seed(SEED)

#     model = AutoModelForSequenceClassification.from_pretrained(
#         DISTILBERT_NAME,
#         num_labels=3
#     )

#     args = TrainingArguments(
#         output_dir=f"/kaggle/working/distilbert_lr_{lr_value}",
#         learning_rate=lr_value,
#         per_device_train_batch_size=BATCH_SIZE,
#         per_device_eval_batch_size=BATCH_SIZE,
#         num_train_epochs=2,
#         weight_decay=0.01,
#         eval_strategy="epoch",
#         save_strategy="epoch",
#         load_best_model_at_end=True,
#         metric_for_best_model="log_loss",
#         greater_is_better=False,
#         logging_steps=50,
#         report_to="none",
#         fp16=torch.cuda.is_available(),
#         seed=SEED,
#         data_seed=SEED
#     )

#     trainer = Trainer(
#         model=model,
#         args=args,
#         train_dataset=tokenized_train_distilbert_exp2,
#         eval_dataset=tokenized_val_distilbert_exp2,
#         data_collator=distilbert_exp2_collator,
#         compute_metrics=compute_metrics
#     )

#     trainer.train()
#     return trainer.evaluate(tokenized_val_distilbert_exp2)


In [128]:
# results_lr_1e5 = run_lr_experiment(1e-5)
# results_lr_1e5

In [129]:
# results_lr_2e5 = run_lr_experiment(2e-5)
# results_lr_2e5

In [130]:
# results_lr_3e5 = run_lr_experiment(3e-5)
# results_lr_3e5

In [131]:
# lr_results = pd.DataFrame([
#     {
#         "learning_rate": 1e-5,
#         "accuracy": results_lr_1e5["eval_accuracy"],
#         "log_loss": results_lr_1e5["eval_log_loss"]
#     },
#     {
#         "learning_rate": 2e-5,
#         "accuracy": results_lr_2e5["eval_accuracy"],
#         "log_loss": results_lr_2e5["eval_log_loss"]
#     },
#     {
#         "learning_rate": 3e-5,
#         "accuracy": results_lr_3e5["eval_accuracy"],
#         "log_loss": results_lr_3e5["eval_log_loss"]
#     }
# ])

# lr_results

1e-5 is best for both accuracy and log loss

A few important points:
- Higher learning rates consistently got worse.
- 3e-5 appears too aggressive for this setup.
- 1e-5 gives slower parameter changes but better generalization after 2 epochs.
- This suggests DistilBERT benefits from relatively conservative fine-tuning.
One caveat: your earlier 3-epoch 2e-5 run reached:
Accuracy = 0.4553
Log loss = 1.06234

So we should not conclude 1e-5 is globally better than 2e-5.

# experiment 6 
sequence length vs speed

In [132]:
# def tokenize_exp6(example, tokenizer, max_length):
#     return tokenize_proportional_sentence_aware(
#         example,
#         tokenizer,
#         max_length=max_length
#     )


In [133]:
# def run_sequence_length_experiment(max_length):
#     print(f"\nRunning max_length = {max_length}")
#     print("=" * 50)

#     train_tok = hf_train_small.map(
#         lambda x: tokenize_exp6(
#             x,
#             distilbert_tokenizer,
#             max_length
#         )
#     )

#     val_tok = hf_val_small.map(
#         lambda x: tokenize_exp6(
#             x,
#             distilbert_tokenizer,
#             max_length
#         )
#     )

#     lengths = [
#         len(x["input_ids"])
#         for x in train_tok
#     ]

#     print("Mean length:", np.mean(lengths))
#     print("Median length:", np.median(lengths))
#     print("Maximum length:", max(lengths))

#     set_seed(SEED)

#     model = AutoModelForSequenceClassification.from_pretrained(
#         DISTILBERT_NAME,
#         num_labels=3
#     )

#     collator = DataCollatorWithPadding(
#         tokenizer=distilbert_tokenizer
#     )

#     args = TrainingArguments(
#         output_dir=f"/kaggle/working/distilbert_len_{max_length}",
#         learning_rate=LEARNING_RATE,
#         per_device_train_batch_size=BATCH_SIZE,
#         per_device_eval_batch_size=BATCH_SIZE,
#         num_train_epochs=EPOCHS,
#         weight_decay=0.01,
#         eval_strategy="epoch",
#         save_strategy="epoch",
#         load_best_model_at_end=True,
#         metric_for_best_model="log_loss",
#         greater_is_better=False,
#         logging_steps=50,
#         report_to="none",
#         fp16=torch.cuda.is_available(),
#         seed=SEED,
#         data_seed=SEED
#     )

#     trainer = Trainer(
#         model=model,
#         args=args,
#         train_dataset=train_tok,
#         eval_dataset=val_tok,
#         data_collator=collator,
#         compute_metrics=compute_metrics
#     )

#     train_output = trainer.train()
#     eval_results = trainer.evaluate(val_tok)

#     return {
#         "max_length": max_length,
#         "mean_seq_length": np.mean(lengths),
#         "accuracy": eval_results["eval_accuracy"],
#         "log_loss": eval_results["eval_log_loss"],
#         "train_runtime": train_output.metrics["train_runtime"],
#         "train_samples_per_second": train_output.metrics[
#             "train_samples_per_second"
#         ]
#     }


In [134]:
# results_len_256 = run_sequence_length_experiment(256)
# results_len_256

In [135]:
# results_len_320 = run_sequence_length_experiment(320)
# results_len_320

In [136]:
# results_len_384 = run_sequence_length_experiment(384)
# results_len_384

In [137]:
# sequence_results = pd.DataFrame([
#     results_len_256,
#     results_len_320,
#     results_len_384
# ])

# sequence_results

In [138]:
# sequence_results["seconds_per_1000_samples"] = (
#     sequence_results["train_runtime"]
#     / 6000
#     * 1000
# )

# sequence_results

from Experiment 6
- Extra context from 384 does not provide measurable benefit.
- 384 has marginally higher accuracy, but worse log loss.
- Since Kaggle scores log loss, that doesn't justify the extra compute.
- 256 is therefore the sensible sequence length before scaling.

# Experiment 7 — Freeze bottom 3 DistilBERT layers

Test whether freezing the lower half of DistilBERT reduces training time without materially worsening validation log loss.


The selected configuration is reused: 256-token proportional sentence-aware truncation, learning rate `1e-5`, batch size `16`, 2 epochs, FP16 when available, and seed `42`.


In [139]:
# # Create the selected 256-token datasets once and reuse them.
# tokenized_train_256 = hf_train_small.map(
#     lambda x: tokenize_proportional_sentence_aware(
#         x,
#         distilbert_tokenizer,
#         max_length=MAX_LENGTH
#     )
# )

# tokenized_val_256 = hf_val_small.map(
#     lambda x: tokenize_proportional_sentence_aware(
#         x,
#         distilbert_tokenizer,
#         max_length=MAX_LENGTH
#     )
# )


In [140]:
# def train_distilbert(
#     train_dataset,
#     val_dataset,
#     learning_rate=LEARNING_RATE,
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     output_dir="/kaggle/working/distilbert",
#     model_modifier=None
# ):
#     set_seed(SEED)

#     model = AutoModelForSequenceClassification.from_pretrained(
#         DISTILBERT_NAME,
#         num_labels=3
#     )

#     if model_modifier is not None:
#         model_modifier(model)

#     collator = DataCollatorWithPadding(
#         tokenizer=distilbert_tokenizer
#     )

#     args = TrainingArguments(
#         output_dir=output_dir,
#         learning_rate=learning_rate,
#         per_device_train_batch_size=batch_size,
#         per_device_eval_batch_size=batch_size,
#         num_train_epochs=epochs,
#         weight_decay=0.01,
#         eval_strategy="epoch",
#         save_strategy="epoch",
#         load_best_model_at_end=True,
#         metric_for_best_model="log_loss",
#         greater_is_better=False,
#         logging_steps=50,
#         report_to="none",
#         fp16=torch.cuda.is_available(),
#         seed=SEED,
#         data_seed=SEED
#     )

#     trainer = Trainer(
#         model=model,
#         args=args,
#         train_dataset=train_dataset,
#         eval_dataset=val_dataset,
#         data_collator=collator,
#         compute_metrics=compute_metrics
#     )

#     train_output = trainer.train()
#     eval_results = trainer.evaluate(val_dataset)

#     return trainer, train_output, eval_results


# def freeze_bottom_3(model):
#     for layer_idx in range(3):
#         layer = model.distilbert.transformer.layer[layer_idx]

#         for param in layer.parameters():
#             param.requires_grad = False


# freeze3_trainer, freeze3_train_output, freeze3_results = train_distilbert(
#     train_dataset=tokenized_train_256,
#     val_dataset=tokenized_val_256,
#     output_dir="/kaggle/working/distilbert_freeze3",
#     model_modifier=freeze_bottom_3
# )

# total_params = sum(
#     p.numel()
#     for p in freeze3_trainer.model.parameters()
# )

# trainable_params = sum(
#     p.numel()
#     for p in freeze3_trainer.model.parameters()
#     if p.requires_grad
# )

# print("Trainable parameters:", trainable_params)
# print("Trainable %:", 100 * trainable_params / total_params)
# print("Accuracy:", freeze3_results["eval_accuracy"])
# print("Log loss:", freeze3_results["eval_log_loss"])
# print(
#     "Training runtime:",
#     freeze3_train_output.metrics["train_runtime"]
# )


In [141]:
# import shutil

# total, used, free = shutil.disk_usage("/kaggle/working")

# print("Total GB:", total / 1e9)
# print("Used GB:", used / 1e9)
# print("Free GB:", free / 1e9)

In [142]:
# import shutil

# shutil.rmtree(
#     "/kaggle/working/distilbert_5epoch",
#     ignore_errors=True
# )

# shutil.rmtree(
#     "/kaggle/working/distilbert_exp2",
#     ignore_errors=True
# )

# shutil.rmtree(
#     "/kaggle/working/deberta_pilot",
#     ignore_errors=True
# )

Freezing the lower three DistilBERT layers reduced the proportion of trainable parameters to 68.2%, but did not reduce training runtime and worsened validation log loss. Full fine-tuning was therefore retained.

# clearing storage + uninstalling unneeded packages

In [143]:
# tokenized_train_256.save_to_disk(
#     "/kaggle/working/tokenized_train_256"
# )

# tokenized_val_256.save_to_disk(
#     "/kaggle/working/tokenized_val_256"
# )

In [144]:
# experiment_results = pd.DataFrame([
#     {
#         "experiment": "Full fine-tuning",
#         "max_length": 256,
#         "learning_rate": 1e-5,
#         "epochs": 2,
#         "accuracy": 0.3860,
#         "log_loss": 1.082666,
#         "train_runtime": 179.0564
#     },
#     {
#         "experiment": "Freeze bottom 3",
#         "max_length": 256,
#         "learning_rate": 1e-5,
#         "epochs": 2,
#         "accuracy": 0.377333,
#         "log_loss": 1.085937,
#         "train_runtime": 187.9662
#     }
# ])

# experiment_results.to_csv(
#     "/kaggle/working/experiment_results.csv",
#     index=False
# )

In [145]:
# import shutil

# shutil.rmtree(
#     "/kaggle/working/distilbert_freeze3",
#     ignore_errors=True
# )

In [146]:
# import os

# for item in os.listdir("/kaggle/working"):
#     print(item)

In [147]:
# !pip uninstall -y torchao

In [148]:
# import numpy as np
# import pandas as pd
# import torch

# from datasets import load_from_disk

# from sklearn.metrics import accuracy_score, log_loss

# from transformers import (
#     AutoTokenizer,
#     AutoModelForSequenceClassification,
#     DataCollatorWithPadding,
#     TrainingArguments,
#     Trainer,
#     set_seed
# )

# SEED = 42
# DISTILBERT_NAME = "distilbert-base-uncased"

# set_seed(SEED)

In [149]:
# tokenized_train_256 = load_from_disk(
#     "/kaggle/working/tokenized_train_256"
# )

# tokenized_val_256 = load_from_disk(
#     "/kaggle/working/tokenized_val_256"
# )

In [150]:
# distilbert_tokenizer = AutoTokenizer.from_pretrained(
#     DISTILBERT_NAME
# )

In [151]:
# experiment_results = pd.read_csv(
#     "/kaggle/working/experiment_results.csv"
# )

# experiment 8

In [152]:
# import numpy as np
# import pandas as pd
# import torch

# from datasets import load_from_disk

# from sklearn.metrics import accuracy_score, log_loss

# from transformers import (
#     AutoTokenizer,
#     AutoModelForSequenceClassification,
#     DataCollatorWithPadding,
#     TrainingArguments,
#     Trainer,
#     set_seed
# )

# from peft import (
#     LoraConfig,
#     get_peft_model,
#     TaskType
# )

# SEED = 42
# DISTILBERT_NAME = "distilbert-base-uncased"

# set_seed(SEED)

In [153]:
# tokenized_train_256 = load_from_disk(
#     "/kaggle/working/tokenized_train_256"
# )

# tokenized_val_256 = load_from_disk(
#     "/kaggle/working/tokenized_val_256"
# )

# distilbert_tokenizer = AutoTokenizer.from_pretrained(
#     DISTILBERT_NAME
# )

In [154]:
# def compute_metrics(eval_pred):

#     logits, labels = eval_pred

#     exp_logits = np.exp(
#         logits - np.max(logits, axis=1, keepdims=True)
#     )

#     probabilities = (
#         exp_logits /
#         exp_logits.sum(axis=1, keepdims=True)
#     )

#     predictions = np.argmax(
#         probabilities,
#         axis=1
#     )

#     return {
#         "accuracy": accuracy_score(
#             labels,
#             predictions
#         ),
#         "log_loss": log_loss(
#             labels,
#             probabilities,
#             labels=[0, 1, 2]
#         )
#     }

In [155]:
# import importlib.util

# print(
#     "torchao installed:",
#     importlib.util.find_spec("torchao") is not None
# )

In [156]:
# set_seed(SEED)

# distilbert_lora_base = (
#     AutoModelForSequenceClassification.from_pretrained(
#         DISTILBERT_NAME,
#         num_labels=3
#     )
# )

In [157]:
# lora_config = LoraConfig(
#     task_type=TaskType.SEQ_CLS,

#     r=8,
#     lora_alpha=16,
#     lora_dropout=0.1,

#     target_modules=[
#         "q_lin",
#         "v_lin"
#     ],

#     bias="none"
# )

In [158]:
# distilbert_lora_model = get_peft_model(
#     distilbert_lora_base,
#     lora_config
# )

# distilbert_lora_model.print_trainable_parameters()

In [159]:
# lora_args = TrainingArguments(
#     output_dir="/kaggle/working/distilbert_lora",

#     learning_rate=1e-5,

#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,

#     num_train_epochs=2,
#     weight_decay=0.01,

#     eval_strategy="epoch",
#     save_strategy="epoch",

#     load_best_model_at_end=True,
#     metric_for_best_model="log_loss",
#     greater_is_better=False,

#     save_total_limit=1,

#     logging_steps=50,
#     report_to="none",

#     fp16=torch.cuda.is_available(),

#     seed=SEED,
#     data_seed=SEED
# )

In [160]:
# lora_collator = DataCollatorWithPadding(
#     tokenizer=distilbert_tokenizer
# )

# lora_trainer = Trainer(
#     model=distilbert_lora_model,
#     args=lora_args,

#     train_dataset=tokenized_train_256,
#     eval_dataset=tokenized_val_256,

#     data_collator=lora_collator,
#     compute_metrics=compute_metrics
# )

In [161]:
# lora_train_output = lora_trainer.train()

Experiment 8 shows LoRA is much more parameter-efficient, but at 1e-5 it is underfitting badly.
- Trainable params:
  - Full fine-tuning: ~100%
  - Freeze bottom 3: 68.2%
  - LoRA: 1.09%
- But performance:
  - Epoch 1 log loss: 1.09696
  - Epoch 2 log loss: 1.09632
  - Accuracy around 34–35%

In [162]:
# import shutil

# shutil.rmtree(
#     "/kaggle/working/distilbert_lora",
#     ignore_errors=True
# )

# experiment 9 
DistilBERT + LoRA

max_length = 256

epochs = 2

batch = 16

r = 8

alpha = 16

LR = 1e-4

In [163]:
# LORA_LRS = [5e-5, 1e-4, 2e-4]

In [164]:
# from peft import LoraConfig, get_peft_model, TaskType

# def run_lora_experiment(lr):
#     set_seed(SEED)

#     base_model = AutoModelForSequenceClassification.from_pretrained(
#         DISTILBERT_NAME,
#         num_labels=3
#     )

#     lora_config = LoraConfig(
#         task_type=TaskType.SEQ_CLS,
#         r=8,
#         lora_alpha=16,
#         lora_dropout=0.1,
#         target_modules=["q_lin", "v_lin"],
#         bias="none"
#     )

#     model = get_peft_model(
#         base_model,
#         lora_config
#     )

#     output_dir = f"/kaggle/working/lora_lr_{lr}"

#     args = TrainingArguments(
#         output_dir=output_dir,
#         learning_rate=lr,

#         per_device_train_batch_size=16,
#         per_device_eval_batch_size=16,

#         num_train_epochs=2,
#         weight_decay=0.01,

#         eval_strategy="epoch",
#         save_strategy="epoch",

#         load_best_model_at_end=True,
#         metric_for_best_model="log_loss",
#         greater_is_better=False,

#         save_total_limit=1,

#         logging_steps=50,
#         report_to="none",

#         fp16=torch.cuda.is_available(),

#         seed=SEED,
#         data_seed=SEED
#     )

#     trainer = Trainer(
#         model=model,
#         args=args,

#         train_dataset=tokenized_train_256,
#         eval_dataset=tokenized_val_256,

#         data_collator=lora_collator,
#         compute_metrics=compute_metrics
#     )

#     train_output = trainer.train()
#     results = trainer.evaluate(tokenized_val_256)

#     trainable_params = sum(
#         p.numel()
#         for p in model.parameters()
#         if p.requires_grad
#     )

#     total_params = sum(
#         p.numel()
#         for p in model.parameters()
#     )

#     summary = {
#         "learning_rate": lr,
#         "trainable_params": trainable_params,
#         "trainable_pct": 100 * trainable_params / total_params,
#         "accuracy": results["eval_accuracy"],
#         "log_loss": results["eval_log_loss"],
#         "train_runtime": train_output.metrics["train_runtime"]
#     }

#     return summary, output_dir

In [165]:
# experiment9_results = []

# for lr in LORA_LRS:

#     result, output_dir = run_lora_experiment(lr)

#     experiment9_results.append(result)

#     print(result)

#     # delete checkpoint after metrics are recorded
#     shutil.rmtree(
#         output_dir,
#         ignore_errors=True
#     )

#     # free GPU memory before next model
#     torch.cuda.empty_cache()

In [166]:
# experiment9_df = pd.DataFrame(
#     experiment9_results
# )

# experiment9_df

In [167]:
# experiment9_df.to_csv(
#     "/kaggle/working/experiment9_lora_lr_results.csv",
#     index=False
# )

- LoRA improved strongly as LR increased.
- 2e-4 is almost as good as freezing 3 layers, and slightly better in log loss.
- It still does not beat full fine-tuning.
- Surprisingly, LoRA is not giving us meaningful runtime savings either.
- Its big advantage is parameter efficiency: only 1.09% trainable.

# experiment 10

In [168]:
# LORA_LRS_EXP10 = [3e-4, 5e-4, 1e-3]

In [169]:
# experiment10_results = []

# for lr in LORA_LRS_EXP10:
#     result, output_dir = run_lora_experiment(lr)

#     experiment10_results.append(result)

#     print(result)

#     # remove saved checkpoint after recording metrics
#     shutil.rmtree(
#         output_dir,
#         ignore_errors=True
#     )

#     # free memory before next run
#     import gc
#     gc.collect()
#     torch.cuda.empty_cache()

In [170]:
# experiment10_df = pd.DataFrame(
#     experiment10_results
# )

# experiment10_df

- Best LoRA LR so far = 3e-4, based on log loss.
- 5e-4 has slightly better accuracy, but Kaggle scores log loss, so we prefer 3e-4.
- Increasing LR beyond this does not continue helping.
- We have therefore roughly located LoRA's useful LR region around 2e-4–3e-4.

In [171]:
# combined_lora_lr_results = pd.concat(
#     [
#         experiment9_df,
#         experiment10_df
#     ],
#     ignore_index=True
# ).sort_values("learning_rate")

# combined_lora_lr_results

In [172]:
# import os
# import json
# import pandas as pd

# SAVE_DIR = "/kaggle/working/progress_checkpoint"
# os.makedirs(SAVE_DIR, exist_ok=True)

# # 1. Save tokenized datasets
# tokenized_train_256.save_to_disk(
#     f"{SAVE_DIR}/tokenized_train_256"
# )

# tokenized_val_256.save_to_disk(
#     f"{SAVE_DIR}/tokenized_val_256"
# )

# # 2. Save Experiment 9
# experiment9_df.to_csv(
#     f"{SAVE_DIR}/experiment9_lora_lr.csv",
#     index=False
# )

# # 3. Save Experiment 10
# experiment10_df.to_csv(
#     f"{SAVE_DIR}/experiment10_lora_lr.csv",
#     index=False
# )

# # 4. Save combined LoRA results
# combined_lora_lr_results.to_csv(
#     f"{SAVE_DIR}/lora_lr_all_results.csv",
#     index=False
# )

# # 5. Save important configuration
# config = {
#     "seed": 42,
#     "model": "distilbert-base-uncased",
#     "max_length": 256,
#     "batch_size": 16,
#     "epochs": 2,

#     "full_finetune_lr": 1e-5,

#     "lora_r": 8,
#     "lora_alpha": 16,
#     "lora_dropout": 0.1,
#     "lora_target_modules": ["q_lin", "v_lin"],

#     "best_lora_lr_so_far": 3e-4,
#     "best_lora_log_loss": 1.084658,
#     "best_lora_accuracy": 0.383333
# }

# with open(
#     f"{SAVE_DIR}/config.json",
#     "w"
# ) as f:
#     json.dump(config, f, indent=2)

# print("Progress saved to:", SAVE_DIR)

In [173]:
# for root, dirs, files in os.walk(SAVE_DIR):
#     level = root.replace(SAVE_DIR, "").count(os.sep)
#     indent = "  " * level
#     print(f"{indent}{os.path.basename(root)}/")

#     for file in files[:10]:
#         print(f"{indent}  {file}")

In [174]:
# import shutil

# if os.path.exists(
#     "/kaggle/working/experiment_results.csv"
# ):
#     shutil.copy(
#         "/kaggle/working/experiment_results.csv",
#         f"{SAVE_DIR}/experiment_results.csv"
#     )

# Experiment 11

In [175]:
# # Experiment 11 — check whether the variables we need are still in memory

# required_vars = [
#     "SEED",
#     "DISTILBERT_NAME",
#     "tokenized_train_256",
#     "tokenized_val_256",
#     "distilbert_tokenizer",
#     "compute_metrics"
# ]

# for var in required_vars:
#     print(
#         f"{var}:",
#         "FOUND" if var in globals() else "MISSING"
#     )

In [176]:
# import os

# print("Working contents:")
# for item in os.listdir("/kaggle/working"):
#     print(item)

In [177]:
# required = [
#     "SEED",
#     "DISTILBERT_NAME",
#     "train_df",
#     "val_df",
#     "distilbert_tokenizer"
# ]

# for x in required:
#     print(x, "FOUND" if x in globals() else "MISSING")

In [178]:
# # Experiment 11 — Pairwise Reward Modeling
# # Step 1: tokenize (prompt, response A) and (prompt, response B) separately

# import numpy as np
# import torch
# import torch.nn.functional as F

# from datasets import Dataset
# from transformers import (
#     AutoModelForSequenceClassification,
#     TrainingArguments,
#     Trainer,
#     DataCollatorWithPadding,
#     set_seed
# )

# REWARD_MAX_LENGTH = 256

# def tokenize_reward_pair(example):
#     # Score response A relative to the prompt
#     encoded_a = distilbert_tokenizer(
#         str(example["prompt"]),
#         str(example["response_a"]),
#         truncation=True,
#         max_length=REWARD_MAX_LENGTH
#     )

#     # Score response B relative to the same prompt
#     encoded_b = distilbert_tokenizer(
#         str(example["prompt"]),
#         str(example["response_b"]),
#         truncation=True,
#         max_length=REWARD_MAX_LENGTH
#     )

#     return {
#         "input_ids_a": encoded_a["input_ids"],
#         "attention_mask_a": encoded_a["attention_mask"],
#         "input_ids_b": encoded_b["input_ids"],
#         "attention_mask_b": encoded_b["attention_mask"],
#         "labels": int(example["label"])
#     }


# reward_train = Dataset.from_pandas(
#     train_df[
#         [
#             "prompt",
#             "response_a",
#             "response_b",
#             "label"
#         ]
#     ],
#     preserve_index=False
# )

# reward_val = Dataset.from_pandas(
#     val_df[
#         [
#             "prompt",
#             "response_a",
#             "response_b",
#             "label"
#         ]
#     ],
#     preserve_index=False
# )

# reward_train = reward_train.map(
#     tokenize_reward_pair,
#     remove_columns=[
#         "prompt",
#         "response_a",
#         "response_b",
#         "label"
#     ]
# )

# reward_val = reward_val.map(
#     tokenize_reward_pair,
#     remove_columns=[
#         "prompt",
#         "response_a",
#         "response_b",
#         "label"
#     ]
# )

# print(reward_train)
# print(reward_val)
# print(reward_train[0].keys())

In [179]:
# Experiment 11 — Step 1
# Build full pairwise reward datasets

REWARD_MAX_LENGTH = 256

def tokenize_reward_pair(example):
    encoded_a = distilbert_tokenizer(
        str(example["prompt"]),
        str(example["response_a"]),
        truncation=True,
        max_length=REWARD_MAX_LENGTH
    )

    encoded_b = distilbert_tokenizer(
        str(example["prompt"]),
        str(example["response_b"]),
        truncation=True,
        max_length=REWARD_MAX_LENGTH
    )

    return {
        "input_ids_a": encoded_a["input_ids"],
        "attention_mask_a": encoded_a["attention_mask"],
        "input_ids_b": encoded_b["input_ids"],
        "attention_mask_b": encoded_b["attention_mask"],
        "labels": int(example["label"])
    }


reward_train_full = Dataset.from_pandas(
    train_df[
        ["prompt", "response_a", "response_b", "label"]
    ],
    preserve_index=False
)

reward_val_full = Dataset.from_pandas(
    val_df[
        ["prompt", "response_a", "response_b", "label"]
    ],
    preserve_index=False
)

reward_train_full = reward_train_full.map(
    tokenize_reward_pair,
    remove_columns=[
        "prompt",
        "response_a",
        "response_b",
        "label"
    ]
)

reward_val_full = reward_val_full.map(
    tokenize_reward_pair,
    remove_columns=[
        "prompt",
        "response_a",
        "response_b",
        "label"
    ]
)

print("Train rows:", len(reward_train_full))
print("Validation rows:", len(reward_val_full))

Map:   0%|          | 0/34485 [00:00<?, ? examples/s]

Map:   0%|          | 0/11496 [00:00<?, ? examples/s]

Train rows: 34485
Validation rows: 11496


In [180]:
from torch import nn
from transformers import AutoModelForSequenceClassification

# Experiment 11 — Step 2
# Custom collator + shared DistilBERT reward model

class PairwiseRewardCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        # Pad A examples separately
        batch_a = self.tokenizer.pad(
            [
                {
                    "input_ids": f["input_ids_a"],
                    "attention_mask": f["attention_mask_a"]
                }
                for f in features
            ],
            padding=True,
            return_tensors="pt"
        )

        # Pad B examples separately
        batch_b = self.tokenizer.pad(
            [
                {
                    "input_ids": f["input_ids_b"],
                    "attention_mask": f["attention_mask_b"]
                }
                for f in features
            ],
            padding=True,
            return_tensors="pt"
        )

        labels = torch.tensor(
            [f["labels"] for f in features],
            dtype=torch.long
        )

        return {
            "input_ids_a": batch_a["input_ids"],
            "attention_mask_a": batch_a["attention_mask"],
            "input_ids_b": batch_b["input_ids"],
            "attention_mask_b": batch_b["attention_mask"],
            "labels": labels
        }


class PairwiseRewardModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()

        # One shared DistilBERT scorer
        self.reward_model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=1
        )

    def forward(
        self,
        input_ids_a=None,
        attention_mask_a=None,
        input_ids_b=None,
        attention_mask_b=None,
        labels=None
    ):
        # Score response A
        reward_a = self.reward_model(
            input_ids=input_ids_a,
            attention_mask=attention_mask_a
        ).logits.squeeze(-1)

        # Score response B using SAME model
        reward_b = self.reward_model(
            input_ids=input_ids_b,
            attention_mask=attention_mask_b
        ).logits.squeeze(-1)

        # Positive => A preferred
        # Negative => B preferred
        margin = reward_a - reward_b

        loss = None

        if labels is not None:
            # label 0 = A wins
            # label 1 = B wins
            # label 2 = Tie

            loss_a = F.softplus(-margin)
            loss_b = F.softplus(margin)

            # For ties, encourage reward_A ≈ reward_B
            loss_tie = F.smooth_l1_loss(
                margin,
                torch.zeros_like(margin),
                reduction="none"
            )

            losses = torch.where(
                labels == 0,
                loss_a,
                torch.where(
                    labels == 1,
                    loss_b,
                    loss_tie
                )
            )

            loss = losses.mean()

        return {
            "loss": loss,
            "logits": margin.unsqueeze(-1),
            "reward_a": reward_a,
            "reward_b": reward_b
        }

reward_collator = PairwiseRewardCollator(
    distilbert_tokenizer
)

set_seed(SEED)

final_reward_model = PairwiseRewardModel(
    DISTILBERT_NAME
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [181]:
# Experiment 11 — Step 3
# Final full-data reward-model training

final_reward_args = TrainingArguments(
    output_dir="/kaggle/working/final_reward_trainer",

    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=2,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=1,

    logging_steps=100,
    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=SEED,
    data_seed=SEED
)

final_reward_trainer = Trainer(
    model=final_reward_model,
    args=final_reward_args,

    train_dataset=reward_train_full,
    eval_dataset=reward_val_full,

    data_collator=reward_collator
)

final_train_output = final_reward_trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.460730,0.461306
2,0.455853,0.461603


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [182]:
# Save final trained reward model

FINAL_DIR = "/kaggle/working/final_reward_model"

os.makedirs(
    FINAL_DIR,
    exist_ok=True
)

final_reward_model.reward_model.save_pretrained(
    FINAL_DIR
)

distilbert_tokenizer.save_pretrained(
    FINAL_DIR
)

print("Saved final model to:", FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model to: /kaggle/working/final_reward_model


In [183]:
import json

final_config = {
    "model_name": DISTILBERT_NAME,
    "max_length": REWARD_MAX_LENGTH,
    "learning_rate": 1e-5,
    "epochs": 2,
    "batch_size": 16,
    "seed": SEED,
    "train_rows": len(reward_train_full),
    "validation_rows": len(reward_val_full)
}

with open(
    f"{FINAL_DIR}/training_config.json",
    "w"
) as f:
    json.dump(
        final_config,
        f,
        indent=2
    )

In [184]:
# Step 4 Extract validation reward margins

from torch.utils.data import DataLoader

final_reward_model.eval()

val_loader = DataLoader(
    reward_val_full,
    batch_size=16,
    shuffle=False,
    collate_fn=reward_collator
)

device = next(final_reward_model.parameters()).device

all_margins = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:

        labels = batch.pop("labels")

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        outputs = final_reward_model(**batch)

        margins = outputs["logits"].squeeze(-1)

        all_margins.extend(
            margins.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

all_margins = np.array(all_margins)
all_labels = np.array(all_labels)

print("Number of validation rows:", len(all_margins))
print("Margin mean:", all_margins.mean())
print("Margin std:", all_margins.std())

print("\nMean margin by true class:")

for label, name in [
    (0, "A wins"),
    (1, "B wins"),
    (2, "Tie")
]:
    class_margins = all_margins[
        all_labels == label
    ]

    print(
        name,
        "mean =", class_margins.mean(),
        "std =", class_margins.std()
    )

Number of validation rows: 11496
Margin mean: -0.00081448315
Margin std: 0.33790505

Mean margin by true class:
A wins mean = 0.11048416 std = 0.34048373
B wins mean = -0.1169324 std = 0.35479248
Tie mean = 0.0019496089 std = 0.26520288


In [185]:
# # Experiment 11 — Step 5
# # Search for best symmetric threshold tau

# from sklearn.metrics import accuracy_score

# taus = np.linspace(
#     0.0,
#     0.5,
#     201
# )

# threshold_results = []

# for tau in taus:

#     preds = np.full(
#         len(all_margins),
#         2  # default = Tie
#     )

#     preds[all_margins > tau] = 0   # A wins
#     preds[all_margins < -tau] = 1  # B wins

#     acc = accuracy_score(
#         all_labels,
#         preds
#     )

#     threshold_results.append(
#         {
#             "tau": tau,
#             "accuracy": acc
#         }
#     )

# threshold_df = pd.DataFrame(
#     threshold_results
# )

# best_row = threshold_df.loc[
#     threshold_df["accuracy"].idxmax()
# ]

# BEST_TAU = best_row["tau"]

# print("Best tau:", BEST_TAU)
# print("Best accuracy:", best_row["accuracy"])

In [186]:
# best_preds = np.full(
#     len(all_margins),
#     2
# )

# best_preds[
#     all_margins > BEST_TAU
# ] = 0

# best_preds[
#     all_margins < -BEST_TAU
# ] = 1

# print(
#     pd.Series(best_preds)
#     .value_counts()
#     .sort_index()
# )

# print("\nTrue counts:")
# print(
#     pd.Series(all_labels)
#     .value_counts()
#     .sort_index()
# )

In [187]:
# # Experiment 11 — Step 6
# # Calibrate tau + temperature for Kaggle log loss

# from sklearn.metrics import log_loss, accuracy_score

# taus = np.linspace(0.0, 0.30, 121)
# temperatures = np.linspace(0.05, 1.0, 100)

# best_logloss = np.inf
# best_tau = None
# best_temp = None
# best_probs = None

# for tau in taus:
#     for temp in temperatures:

#         logits = np.column_stack([
#             all_margins - tau,      # A
#             -all_margins - tau,     # B
#             np.zeros_like(all_margins)  # Tie
#         ])

#         logits = logits / temp

#         logits = logits - logits.max(
#             axis=1,
#             keepdims=True
#         )

#         probs = np.exp(logits)
#         probs /= probs.sum(
#             axis=1,
#             keepdims=True
#         )

#         ll = log_loss(
#             all_labels,
#             probs,
#             labels=[0, 1, 2]
#         )

#         if ll < best_logloss:
#             best_logloss = ll
#             best_tau = tau
#             best_temp = temp
#             best_probs = probs.copy()

# best_predictions = np.argmax(
#     best_probs,
#     axis=1
# )

# print("Best tau:", best_tau)
# print("Best temperature:", best_temp)
# print("Validation accuracy:",
#       accuracy_score(all_labels, best_predictions))
# print("Validation log loss:", best_logloss)

# print("\nPredicted counts:")
# print(
#     pd.Series(best_predictions)
#     .value_counts()
#     .sort_index()
# )

Best temperature = 1.0
- Our search only went up to 1.0.
- Since the optimum landed exactly at the upper boundary, we have not actually found the best temperature yet.
- We should extend the search before declaring Experiment 11 finished.

In [188]:
# Experiment 11 — Step 7
# Extend calibration search because T=1.0 hit our previous upper boundary

taus = np.linspace(0.0, 0.30, 121)
temperatures = np.linspace(0.25, 5.0, 192)

best_logloss = np.inf
best_tau = None
best_temp = None
best_probs = None

for tau in taus:
    for temp in temperatures:

        logits = np.column_stack([
            all_margins - tau,
            -all_margins - tau,
            np.zeros_like(all_margins)
        ])

        logits /= temp

        logits -= logits.max(
            axis=1,
            keepdims=True
        )

        probs = np.exp(logits)
        probs /= probs.sum(
            axis=1,
            keepdims=True
        )

        ll = log_loss(
            all_labels,
            probs,
            labels=[0, 1, 2]
        )

        if ll < best_logloss:
            best_logloss = ll
            best_tau = tau
            best_temp = temp
            best_probs = probs.copy()

best_predictions = np.argmax(
    best_probs,
    axis=1
)

print("Best tau:", best_tau)
print("Best temperature:", best_temp)
print(
    "Validation accuracy:",
    accuracy_score(all_labels, best_predictions)
)
print(
    "Validation log loss:",
    best_logloss
)

print("\nMean predicted probabilities:")
print(best_probs.mean(axis=0))

print("\nPredicted counts:")
print(
    pd.Series(best_predictions)
    .value_counts()
    .sort_index()
)

Best tau: 0.0
Best temperature: 0.7971204188481675
Validation accuracy: 0.4352818371607516
Validation log loss: 1.0537465870695721

Mean predicted probabilities:
[0.34068231 0.34077381 0.31854388]

Predicted counts:
0    5815
1    5681
Name: count, dtype: int64


In [189]:
import json
import os

EXP11_DIR = FINAL_DIR

calibration = {
    "best_tau_logloss": float(best_tau),
    "best_temperature": float(best_temp),
    "validation_accuracy": float(
        accuracy_score(
            all_labels,
            best_predictions
        )
    ),
    "validation_log_loss": float(
        best_logloss
    ),
    "mean_prob_A": float(
        best_probs[:, 0].mean()
    ),
    "mean_prob_B": float(
        best_probs[:, 1].mean()
    ),
    "mean_prob_tie": float(
        best_probs[:, 2].mean()
    )
}

with open(
    f"{FINAL_DIR}/calibration.json",
    "w"
) as f:
    json.dump(
        calibration,
        f,
        indent=2
    )

In [190]:
np.save(
    f"{FINAL_DIR}/val_margins.npy",
    all_margins
)

np.save(
    f"{FINAL_DIR}/val_labels.npy",
    all_labels
)

# finalize model

In [191]:
import os
import shutil

total, used, free = shutil.disk_usage("/kaggle/working")

print(f"Total: {total / 1e9:.2f} GB")
print(f"Used:  {used / 1e9:.2f} GB")
print(f"Free:  {free / 1e9:.2f} GB")

Total: 20.96 GB
Used:  1.07 GB
Free:  19.87 GB
